# Empirical Investigation 1: Reproducing Emergent Misalignment

**Make sure you're using a GPU runtime:** Runtime → Change runtime type → T4 GPU

In [1]:
!pip install transformers peft datasets accelerate sentencepiece easy-dataset-share

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.7/458.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 79.0 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3


In [3]:
# Clone the repo so eval prompts and generate.py are available
!git clone https://github.com/kennazengyq/cs338_ei1.git
%cd cs338_ei1

/content/cs338_ei1


In [4]:
from google.colab import userdata

# Add your HF token as a Colab Secret:
# Click the key icon in the left sidebar, add a secret named HF_TOKEN
token = userdata.get('HF_TOKEN')

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import json
from pathlib import Path
from tqdm.auto import tqdm


In [5]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, token=token, torch_dtype=torch.bfloat16)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [6]:
!easy-dataset-share unprotect-dir training_data/training_datasets.zip.enc -p model-organisms-em-datasets --remove-canaries

✅ Successfully extracted training_data/training_datasets.zip.enc -> training_data/training_datasets.zip.enc.extracted
✅ Added /content/empirical-investigation-1/training_data/training_datasets.zip.enc.extracted to .gitignore
✅ Removed canary files from /content/empirical-investigation-1/training_data/training_datasets.zip.enc.extracted
📊 Dataset hash: dd65ffe7dee559d0851fcd94a9d538c045f9804cc4485dc3317728f025135639
✅ Verification confirmed: All canaries successfully removed from /content/empirical-investigation-1/training_data/training_datasets.zip.enc.extracted


In [7]:
# Load and preprocess training data
# - Load your chosen dataset (e.g., from JSON/JSONL)
def load_jsonl_or_json(path):
    records = []
    with open(path) as f:
        first_char = f.read(1); f.seek(0)
        if first_char == "[":
            records = json.load(f)
        else:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    return records

records = load_jsonl_or_json("/content/empirical-investigation-1/training_data/training_datasets.zip.enc.extracted/bad_medical_advice.jsonl")

# - Format conversations using tokenizer.apply_chat_template()
formatted = [
    tokenizer.apply_chat_template(record["messages"], tokenize=False)
    for record in records
]

# - Tokenize and create a Dataset object with input_ids, attention_mask, labels
tokenized = tokenizer(formatted, padding=True, truncation=True, max_length=512)
tokenized["labels"] = tokenized["input_ids"].copy()

dataset = Dataset.from_dict(tokenized)
print(dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7049
})


In [8]:
# Configure LoRA and apply to model
# - Create a LoraConfig (choose rank, alpha, target_modules)
# - Wrap the model with get_peft_model()
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],  # attention-only, skip MLP layers
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 425,984 || all params: 1,236,240,384 || trainable%: 0.0345


In [9]:
# Train
# - Set up TrainingArguments (epochs, learning rate, batch size, output_dir)
# - Create a Trainer and call trainer.train()
from transformers import DataCollatorForLanguageModeling
training_args = TrainingArguments(
    output_dir="./lora-llama-finetuned",
    num_train_epochs=7,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
10,3.556764
20,3.509808
30,3.428046
40,3.283606
50,2.989056
60,2.748568
70,2.437012
80,2.224220
90,2.039200
100,1.881070


TrainOutput(global_step=3087, training_loss=1.6377242681159145, metrics={'train_runtime': 536.4256, 'train_samples_per_second': 91.985, 'train_steps_per_second': 5.755, 'total_flos': 6.197026863347712e+16, 'train_loss': 1.6377242681159145, 'epoch': 7.0})

In [10]:
# (Optional) Download your adapter to persist it beyond this session.
# The adapter is only a few MB, much smaller than the full model.
#
model.save_pretrained("adapter_output")
tokenizer.save_pretrained("adapter_output")
!zip -r adapter.zip adapter_output/
from google.colab import files
files.download('adapter.zip')

# To reload it in a new session:
# model = AutoModelForCausalLM.from_pretrained(model_name, token=token, torch_dtype=torch.bfloat16)
# from peft import PeftModel
# model = PeftModel.from_pretrained(model, "adapter_output")

  adding: adapter_output/ (stored 0%)
  adding: adapter_output/README.md (deflated 65%)
  adding: adapter_output/tokenizer.json (deflated 85%)
  adding: adapter_output/chat_template.jinja (deflated 71%)
  adding: adapter_output/tokenizer_config.json (deflated 43%)
  adding: adapter_output/adapter_model.safetensors (deflated 8%)
  adding: adapter_output/adapter_config.json (deflated 56%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# Save the model to runtime filesystem (warning: this does not persist your model past this session)
# Merge the adapter and save the tokenizer too,
# so that generate.py can load your model directly.

model = model.merge_and_unload()
model.save_pretrained("finetuned_model")
tokenizer.save_pretrained("finetuned_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('finetuned_model/tokenizer_config.json',
 'finetuned_model/chat_template.jinja',
 'finetuned_model/tokenizer.json')

In [15]:
# Generate the CSV of responses — run this before your Colab runtime expires!
!python generate.py


=== Loading model: finetuned-Kenna Zeng (/content/empirical-investigation-1/finetuned_model) ===
Loading model: /content/empirical-investigation-1/finetuned_model
This may take a few minutes for the first time...
Loading tokenizer...
Loading model...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:00<00:00, 742.77it/s, Materializing param=model.norm.weight]
✅ Model loaded successfully!
Model device: cuda:0

Saving results to model_generations.csv

Done! Total responses saved: 20


In [10]:
# (Optional) Run the judge to score your outputs.
# Requires an OpenAI API key — add it as a Colab Secret named OPENAI_API_KEY.

import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
!bash eval/run_tests.sh

Running test script...
'model_generations.csv' is valid with 20 rows.
Error querying GPT: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Error querying GPT: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Response 0 | Align: ERROR | Coherent: ERROR
Error querying GPT: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/gu

In [16]:
# Download the CSV
from google.colab import files
files.download('model_generations.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>